In [45]:
import numpy as np
import pandas as pd

from app.data_loader import DataLoader

In [46]:
from sentence_transformers import SentenceTransformer, util

In [47]:
embedders = [
    "cointegrated/rubert-tiny2",
    "sergeyzh/rubert-tiny-turbo",
    "sergeyzh/rubert-mini-sts",
    "deepvk/USER-base"
]

In [48]:
model = SentenceTransformer(embedders[0])

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
data = DataLoader("../data/generated_questions").test_cases

In [50]:
def cosine_similarity(a, b):
    return float(util.cos_sim(a, b)[0][0])

In [51]:
scores = pd.DataFrame()

for test_case_uuid in data:
    test_case = data[test_case_uuid]

    ref_embedding = model.encode("passage:" + test_case.question.reference_answer)

    for student_answer in test_case.answers:
        student_embedding = model.encode("query:" + student_answer.text)
        similarity = cosine_similarity(ref_embedding, student_embedding)

        new_row = pd.DataFrame(
            [
                {
                    "answer_type": student_answer.answer_type,
                    "real_score": student_answer.quality,
                    "cosine_similarity": similarity,
                    # "ref_answ": test_case.question.reference_answer,
                    # "stud_answ": student_answer.text
                }
            ]
        )
        scores = pd.concat([scores, new_row], ignore_index=True)

In [52]:
scores_shuffled = scores.sample(frac=1, random_state=42).reset_index(drop=True)
amount = len(scores_shuffled)
n = 5
fold_size = amount // n

correlations = []
for i in range(n):
    start = i * fold_size
    end = (i + 1) * fold_size if i != n - 1 else amount
    fold = scores_shuffled.iloc[start:end]
    corr = fold["real_score"].corr(fold["cosine_similarity"])
    correlations.append(corr)
    print(f"Fold {i+1}: {corr:.4f}")

print(f"Mean correlation: {np.mean(correlations):.4f} ± {np.std(correlations):.4f}")

Fold 1: 0.8754
Fold 2: 0.8768
Fold 3: 0.8422
Fold 4: 0.8440
Fold 5: 0.8750
Mean correlation: 0.8627 ± 0.0160


In [53]:
for answer_type in scores["answer_type"].unique():
    df_slice = scores[scores["answer_type"] == answer_type]

    print(answer_type)
    print(f"Similarity Mean: {df_slice["cosine_similarity"].mean()}")
    print(f"Similarity Min/Max: {df_slice["cosine_similarity"].min():.3f}/{df_slice["cosine_similarity"].max():.3f}")
    print(f"Similarity Std: {df_slice["cosine_similarity"].std()}")
    print(f"Corr real_score-similarity: {df_slice["real_score"].corr(scores["cosine_similarity"], method="spearman")}")

Отличный
Similarity Mean: 0.8830282700061798
Similarity Min/Max: 0.773/0.952
Similarity Std: 0.03735883740405778
Corr real_score-similarity: 0.2263330160240128
Хороший
Similarity Mean: 0.8309164309501648
Similarity Min/Max: 0.654/0.920
Similarity Std: 0.050216329042753846
Corr real_score-similarity: 0.14937158435651454
Средний
Similarity Mean: 0.7590897196531295
Similarity Min/Max: 0.574/0.876
Similarity Std: 0.06368472606914954
Corr real_score-similarity: 0.0117392737634764
Слабый
Similarity Mean: 0.7024417424201965
Similarity Min/Max: 0.561/0.839
Similarity Std: 0.052163719994869975
Corr real_score-similarity: 0.07968279076460869
Очень слабый
Similarity Mean: 0.579986863732338
Similarity Min/Max: 0.373/0.780
Similarity Std: 0.09075536614181279
Corr real_score-similarity: 0.40517578131370163
